# IAM — vLLM serving experiment: Qwen2.5-Coder 3B

Free Colab GPU experiment for OD-41 (WP9/A8). Uses the **same** coding task, prompt and assertions as `tests/model_competition/smoke_coding.py`.

Rules: free resources only, no paid upgrades, results are untrusted until validated by IAM Core, and all artifacts must be copied to IAM-controlled storage before the session ends.

Requires an NVIDIA GPU with compute capability >= 7.5 (T4 or newer). Kaggle's P100 (6.0) is **not** suitable for vLLM.

In [ ]:
import torch, json, time, os, subprocess, threading, requests
assert torch.cuda.is_available(), 'No GPU attached to this Colab session'
name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
capability_float = float(f'{capability[0]}.{capability[1]}')
print({'gpu': name, 'compute_capability': capability_float, 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)})
assert capability_float >= 7.5, f'vLLM requires >= 7.5, got {capability_float}'

In [ ]:
!pip -q install vllm lm-eval peft trl accelerate boto3
print('Install complete. If imports fail, use Runtime -> Restart runtime and re-run the GPU check.')

## Start the OpenAI-compatible server

The task and expected behaviour are copied unchanged from IAM's benchmark harness.

In [ ]:
MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
PORT = 8000
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--port', str(PORT),
    '--max-model-len', '4096', '--dtype', 'float16',
])

TASK = (
    'Write a Python function named normalise_path(path: str) -> str that:\n'
    '1. collapses repeated \u0027/\u0027 characters into a single \u0027/\u0027,\n'
    '2. strips leading and trailing \u0027/\u0027,\n'
    '3. returns \u0027\u0027 when path is None or empty.\n'
    'Return ONLY a single fenced python code block containing the function. '
    'No explanation, no imports, no example calls.'
)

for _ in range(120):
    try:
        r = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=2)
        if r.status_code == 200:
            print('vLLM ready:', r.json()['data'][0]['id'])
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('vLLM server did not become ready')

In [ ]:
def chat(prompt, max_tokens=512):
    started = time.perf_counter()
    body = {'model': MODEL, 'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0, 'max_tokens': max_tokens}
    response = requests.post(f'http://127.0.0.1:{PORT}/v1/chat/completions', json=body, timeout=300)
    response.raise_for_status()
    payload = response.json()
    return {
        'content': payload['choices'][0]['message']['content'],
        'usage': payload.get('usage', {}),
        'latency_ms': round((time.perf_counter() - started) * 1000, 1),
    }

trial = chat(TASK)
print(json.dumps({'latency_ms': trial['latency_ms'], 'usage': trial['usage']}, indent=2))
print(trial['content'])

## Benchmark: repeated + concurrent requests

Record median/min/max latency, peak GPU memory and correctness. This is an experiment, not a deployment decision.

In [ ]:
import statistics
torch.cuda.reset_peak_memory_stats()

warm = [chat(TASK) for _ in range(5)]
latencies = [r['latency_ms'] for r in warm]
peak_vram_mb = round(torch.cuda.max_memory_allocated() / 2**20, 1)
report = {
    'model': MODEL,
    'engine': 'vllm',
    'gpu': name,
    'compute_capability': capability_float,
    'latency_ms': {
        'min': min(latencies), 'median': statistics.median(latencies), 'max': max(latencies)
    },
    'peak_vram_mb': peak_vram_mb,
    'completion_tokens': sum(r['usage'].get('completion_tokens', 0) for r in warm),
    'sample': warm[0]['content'],
}
print(json.dumps(report, indent=2))

## Persist artifacts to IAM-controlled storage

Credentials are read from environment/Colab secrets — never written into the notebook or the report. If S3 settings are absent, the JSON is still saved locally for manual transfer.

In [ ]:
with open('iam_vllm_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

endpoint = os.getenv('IAM_S3_ENDPOINT')
bucket = os.getenv('IAM_S3_BUCKET')
if endpoint and bucket:
    import boto3
    s3 = boto3.client(
        's3', endpoint_url=endpoint,
        aws_access_key_id=os.environ['IAM_S3_ACCESS_KEY'],
        aws_secret_access_key=os.environ['IAM_S3_SECRET_KEY'],
    )
    key = f"experiments/{os.getenv('IAM_EXPERIMENT_ID', 'colab-vllm-1')}/benchmark/vllm_serving_report.json"
    s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(report).encode('utf-8'))
    print('Uploaded to', key)
else:
    print('No IAM S3 settings found; keep iam_vllm_report.json and transfer it manually.')

In [ ]:
server.terminate()
print('vLLM server stopped')